In [7]:
%pip install fastapi uvicorn pydantic


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import uvicorn
import numpy as np
import logging
from pathlib import Path as FSPath 
from typing import List
from joblib import load
import nest_asyncio
from enum import Enum
from fastapi import FastAPI,Response, HTTPException, Path as Param
from fastapi.responses import StreamingResponse
from fastapi.responses import JSONResponse
from pydantic import BaseModel,  conlist, Field
from sklearn.pipeline import Pipeline

In [40]:
# Directorio donde están tus modelos
MODELS_DIR = Path.cwd().parent  / "models"

# Diccionario para alojar pipelines por número de clusters
pipelines: dict[int, joblib.load] = {}

In [ ]:
# Instanciar un logger de módulo
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [59]:
import re
import joblib
from pathlib import Path
from prometheus_client import Counter
from prometheus_client import CollectorRegistry, Counter

# 1. Creamos un registry exclusivo para estas métricas
my_registry = CollectorRegistry()

# 2. Definimos los contadores en ese registry
PIPELINE_LOAD_SUCCESS = Counter(
    "pipeline_load_success_total",
    "Total pipelines cargados con éxito",
    registry=my_registry
)
PIPELINE_LOAD_FAILURE = Counter(
    "pipeline_load_failure_total",
    "Total pipelines con error de carga",
    registry=my_registry
)


for path in MODELS_DIR.glob("pipeline_kmeans*.joblib"):
    # 1. Validar nombre con regex
    m = re.match(r"^pipeline_kmeans(\d+)$", path.stem)
    if not m:
        logger.warning("Ignorando nombre inesperado: %s", path.name)
        continue

    k = int(m.group(1))
    try:
        # 2. Cargar pipeline
        pipelines[k] = joblib.load(path)
        PIPELINE_LOAD_SUCCESS.inc()
        logger.info("Cargado pipeline k=%d desde %s", k, path.name)
    except Exception as err:
        PIPELINE_LOAD_FAILURE.inc()
        logger.error("Error cargando %s: %s", path.name, err, exc_info=True)

# 3. Log final
logger.info("Pipelines disponibles: %s", sorted(pipelines.keys()))


2025-08-25 16:23:32,187 ERROR: Task exception was never retrieved
future: <Task finished name='Task-51' coro=<Server.serve() done, defined at d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\main.py", line 580, in run
    server.run()
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.v

In [ ]:

# Interactuamos con la API usando este elemento
#app = FastAPI(title='Implementando un modelo de Machine Learning')


#----------------------------------------------
@app.exception_handler(Exception)
async def all_exception_handler(request, exc):
    logger.exception("Unhandled exception:")
    return JSONResponse(
        status_code=500,
        content={"detail": str(exc)}
    )
#---------------------------------------------

# Configurar logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------------------------------
# Carga dinámica de todos los pipelines 
# Cargar todos los joblib que tienen el mismo patrón
"""
for path in MODELS_DIR.glob("pipeline_kmeans*.joblib"):
    # Extraemos el número de k del nombre del archivo
    stem = path.stem                    # e.g. "pipeline_kmeans3"
    k = int(stem.replace("pipeline_kmeans", ""))  
    pipelines[k] = load(path)
logger.info("Pipelines cargados: %s", list(pipelines.keys()))
"""
# ------------------------------------------------------------------------------------------
# Modelos de request/response con Pydantic
# Definimos las clases de request/response

class PredictRequest(BaseModel):
    k: int = Field(gt=0, description="Número de clústeres (k) deseado")
    data: list[list[float]] = Field(..., description="Datos a clasificar (lista de observaciones)")


"""
class PredictRequest(BaseModel):
    k: int
    data: List[conlist(float, min_items=2, max_items=2)]
"""
class PredictResponse(BaseModel):
    k: int
    clusters: list[int]
    inertia: float
    
#--------------------------------------------------------------------------------------

# Dependencia para fetch de pipeline
def get_pipeline_or_404(k: int) -> Pipeline:
    pipe = pipelines.get(k)
    if not pipe:
        raise HTTPException(404, f"No existe modelo para k={k}")
    return pipe

# -------------------------------------------------------------------------------------------
# Defenimos un método GET para el endpoint
@app.get("/")
def home():
    return "¡Felicitaciones!, tu API está funcionando según lo esperado. Anda ahora a http://localhost:8000/docs."


# -------------------------------------------------------------------------------------------
# Endpoint de debug para inspeccionar pipelines
@app.get("/debug/pipelines")
def debug_pipelines():
    """
    Devuelve un dict con:
      k -> lista de nombres de pasos en el pipeline
    """
    return {
        k: list(pipe.named_steps.keys())
        for k, pipe in pipelines.items()
    }

# ------------------------------------------------------------------------------------

# Este endpoint maneja la lógica necesaria para clasificar.
# Requiere como entrada el vector de características del viaje y el umbral de confianza para la clasificación.
# Crear el endpoint de predicción
@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest, pipeline: Pipeline = Depends(get_pipeline_or_404)):
    # Log de entrada
    logger.debug("Received /predict request — k: %s, data length: %d", req.k, len(req.data))
    
    # Validación de forma
    X = np.array(req.data)
    if X.ndim != 2:
        raise HTTPException(422, "La entrada debe ser una matriz 2D")
    expected = pipeline.named_steps["kmeans"].n_features_in_
    if X.shape[1] != expected:
        raise HTTPException(422, f"Se esperaban {expected} features, recibidas {X.shape[1]}")

    # Predicción y manejo de errores
    try:
        clusters = pipeline.predict(X)
    except NotFittedError:
        logger.error("Pipeline no entrenado para k=%s", req.k, exc_info=True)
        raise HTTPException(500, "Modelo no listo para predecir")
    except Exception:
        logger.exception("Error inesperado en /predict")
        raise HTTPException(500, "Error interno en el servidor")

    inertia = float(pipeline.named_steps["kmeans"].inertia_)
    return PredictResponse(k=req.k, clusters=clusters.tolist(), inertia=inertia)

2025-08-25 15:07:01,067 INFO: Pipelines cargados: [1, 2, 3, 4, 5, 6, 7]


In [63]:
app = FastAPI(title='Implementando un modelo de Machine Learning')

# pipelines[k] = {"model": Pipeline, "X_pca": np.ndarray}
pipelines: dict[int, dict] = {}

@app.get("/")
async def root():
    return {
        "message": "Service up ✅",
        "endpoints": {
            "list_pipelines": "/pipelines/",
            "plot_cluster": "/plot/{k}",
            "docs": "/docs"
        }
    }


### Endpoint /metrics

In [64]:
from prometheus_client import CollectorRegistry, Counter, generate_latest, CONTENT_TYPE_LATEST
@app.get("/metrics")
async def metrics():
    # Genera el texto con todas las series de `my_registry`
    payload = generate_latest(my_registry)
    # CONTENT_TYPE_LATEST = "text/plain; version=0.0.4; charset=utf-8"
    return Response(payload, media_type=CONTENT_TYPE_LATEST)


In [65]:
@app.on_event("startup")
def load_data_and_pipelines():
    for path in MODELS_DIR.glob("pipeline_kmeans*.joblib"):
        m = re.match(r"^pipeline_kmeans(\d+)$", path.stem)
        if not m:
            logger.warning("Ignorando nombre inesperado: %s", path.name)
            continue

        k = int(m.group(1))
        try:
            pipelines[k] = joblib.load(path)
            PIPELINE_LOAD_SUCCESS.inc()
            logger.info("Cargado pipeline k=%d", k)
        except Exception as err:
            PIPELINE_LOAD_FAILURE.inc()
            logger.error("Error cargando %s: %s", path.name, err, exc_info=True)


C:\Users\crist\AppData\Local\Temp\ipykernel_16284\2874915712.py:1: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


### Endpoint para listar pipelines disponibles

{
  "pipelines": [
    {"k": 1, "filename": "pipeline_kmeans1.joblib"},
    …,
    {"k": 7, "filename": "pipeline_kmeans7.joblib"}
  ]
}


In [66]:
@app.get("/pipelines", summary="Listar modelos K-means disponibles")
async def list_pipelines():
    lista = [{"k": k, "filename": f"pipeline_kmeans{k}.joblib"} 
             for k in sorted(pipelines.keys())]
    return JSONResponse(content={"pipelines": lista})


### Endpoint /plot/{k} para visualizar clusters

In [44]:
from fastapi.responses import StreamingResponse


In [77]:
# 3. Endpoint para generar el plot usando el pipeline ya cargado
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from io import BytesIO
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

app = FastAPI()

# pipelines ya fue poblado en startup: { k: {"model": Pipeline, "X_pca": np.ndarray}, ... }

@app.get("/plot/{k}")
def plot_clusters(k: int):
    entry = pipelines.get(k)
    if entry is None:
        raise HTTPException(status_code=404, detail=f"No existe pipeline para k={k}")

    # Extraer modelo y datos
    kmeans_pipeline = entry["model"]
    X_pca = entry["X_pca"]

    # Predecir etiquetas de cluster
    try:
        labels = kmeans_pipeline.predict(X_pca)
    except Exception as err:
        raise HTTPException(status_code=500, detail=f"Error en predicción: {err}")

    # Construir la figura
    fig, ax = plt.subplots(figsize=(6, 6))
    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        cmap="viridis",
        alpha=0.7
    )
    ax.set_title(f"KMeans k={k}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    # Serializar a PNG
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)

    return StreamingResponse(buf, media_type="image/png")

In [76]:
pipeline = pipelines.get(3)
print(pipeline)

{'model': Pipeline(steps=[('kmeans', KMeans(n_clusters=3, random_state=42))]), 'X_pca': array([[ 0.54803528,  3.14953242],
       [ 1.97075738,  0.0738964 ],
       [ 0.76287803,  0.23995231],
       ...,
       [-0.4982403 , -1.27566034],
       [ 2.30321616,  0.20037049],
       [ 0.72930959, -0.14095568]])}


¡Corriendo la celda que viene echaremos a andar el servidor!

Esto causará que el notebook se bloquee (No podremos correr más celdas) hasta que interrumpamos de forma manual el kernel.
Podemos hacer eso haciendo click en la pestaña **kernel** y luego **Interrupt**.

In [ ]:
logging.basicConfig(level=logging.DEBUG)

# Esto deja correr al servidor en un ambiente interactivo como un jupyter notebook
nest_asyncio.apply()

# Donde se hospedará el servidor
host = "127.0.0.1"

# Iniciamos el servidor
uvicorn.run(app, host= host, port=8000)

INFO:     Started server process [16284]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52672 - "GET /plot/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:61778 - "GET /plot/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:61778 - "GET /plot/4 HTTP/1.1" 200 OK
INFO:     127.0.0.1:61778 - "GET /plot/6 HTTP/1.1" 200 OK


¡El servidor está corriendo! Vamos a http://localhost:8000/ para verlo en acción.

In [49]:
from pathlib import Path

MODELS_DIR = Path.cwd().parent  / "models"         # ajusta a tu ruta real
files = list(MODELS_DIR.glob("pipeline_kmeans*.joblib"))
print("Archivos encontrados:", files)


Archivos encontrados: [WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desarrollo de proyectos y productos de datos/Tareas/SegmentacionClientes/models/pipeline_kmeans1.joblib'), WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desarrollo de proyectos y productos de datos/Tareas/SegmentacionClientes/models/pipeline_kmeans2.joblib'), WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desarrollo de proyectos y productos de datos/Tareas/SegmentacionClientes/models/pipeline_kmeans3.joblib'), WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desarrollo de proyectos y productos de datos/Tareas/SegmentacionClientes/models/pipeline_kmeans4.joblib'), WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desarrollo de proyectos y productos de datos/Tareas/SegmentacionClientes/models/pipeline_kmeans5.joblib'), WindowsPath('d:/crist/OneDrive/Estudios/Mg Data Science_2024/Trimestre 5/Desa